# 교안 01-2: 웹 검색 MCP 서버 붙이기

앞 실습에서는 **우리가 도구를 직접** 불렀습니다. 이번에는 도구를 **에이전트에 넘겨**, 무엇을 언제 부를지 모델이 정하게 합니다.

## 핵심 목표

웹 검색 서버를 에이전트에 붙여, 모델이 스스로 검색하고 근거 URL 까지 밝히게 만든다.

## 학습 순서

1. 검색 서버(duckduckgo-mcp-server) 연결과 도구 두 개(`search`·`fetch_content`)
2. 검색 도구 직접 호출
3. 본문 가져오기
4. 검색 도구를 붙인 에이전트: 근거 URL 까지 답하게 하기

## 쓰는 MCP 서버와 공식 문서

| 서버 | 실행 | 전송 | 공식 문서 |
|---|---|---|---|
| 웹 검색 `duckduckgo-mcp-server` | `uvx` | stdio | https://github.com/nickclyde/duckduckgo-mcp-server |

## 준비물

- **uv**(`uvx --version` 으로 확인). 없으면 https://docs.astral.sh/uv/
- 인터넷. 검색 서버는 **API 키가 필요 없습니다**.
- **에이전트를 만드는 절부터 `OPENAI_API_KEY`** 가 필요합니다(일차 폴더의 `.env`).

> 검색 결과는 매번 달라집니다. 실행할 때마다 문장이 바뀌는 것이 정상입니다.

---
## 준비

같은 폴더의 `utils.py` 도우미를 씁니다. `load_api_key()` 는 일차 폴더와 실습자료 루트의 `.env` 를 읽어
`OPENAI_API_KEY` 가 있는지 **맨 앞에서** 확인합니다.

In [ ]:
import sys
from pathlib import Path

# 노트북에는 __file__ 이 없다. 주피터는 노트북이 있는 폴더를 작업 폴더로 잡아 주므로 그 위가 일차 폴더다.
DAY_DIR = Path.cwd().parent        # 일차 폴더(day21). 아래 경로들의 기준점
sys.path.append(str(DAY_DIR))   # 일차 폴더의 utils.py 를 쓴다

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from utils import block_text, load_api_key, print_trajectory

# 모델을 부르는 절이 있으므로 키를 맨 앞에서 확인한다. 중간에 멈추면 어디까지 됐는지 헷갈린다.
load_api_key(DAY_DIR)
print("준비 완료. 일차 폴더:", DAY_DIR)

---
## 서버 설정: 실행기가 `npx` 가 아니라 `uvx` 인 이유

앞 실습의 파일시스템 서버는 **Node 패키지**라 `npx` 로 띄웠습니다.
이 검색 서버는 **파이썬 패키지**라 파이썬 쪽 실행기인 `uvx` 를 씁니다(`uv` 를 설치하면 함께 깔립니다).

| 키 | 값 | 뜻 |
|---|---|---|
| `command` | `"uvx"` | 서버를 띄울 **실행기**. 파이썬 패키지를 받아 임시 환경에서 실행한다 |
| `args[0]` | `"duckduckgo-mcp-server"` | **띄울 서버 패키지 이름**(PyPI 에 공개된 이름) |
| `transport` | `"stdio"` | 내 컴퓨터에 자식 프로세스로 띄우고 표준입출력으로 대화 |

여기엔 `-y` 같은 옵션도, 허용 폴더 같은 인자도 없습니다. **인자는 서버가 정합니다.**
파일시스템 서버가 폴더를 요구한 것도 그 서버의 규칙이었을 뿐입니다. 새 서버를 붙일 때마다 **그 서버 문서**를 봐야 하는 이유입니다.

In [ ]:
# 웹 검색 서버: 검색 결과 목록과 페이지 본문을 가져오는 도구를 내준다. API 키가 필요 없다.
WEB_SEARCH = {
    "command": "uvx",                       # 파이썬 패키지를 받아 실행하는 실행기(uv 에 딸려 온다)
    "args": ["duckduckgo-mcp-server"],      # 띄울 서버 패키지 이름
    "transport": "stdio",                   # 내 컴퓨터에 프로세스로 띄운다
}

---
## 1. 검색 서버에 붙기

`{"search": WEB_SEARCH}` 의 `"search"` 는 우리가 붙이는 별명입니다.
노트북에서는 `session()` 블록 대신 **`get_tools()`** 를 씁니다(블록이 셀 끝에서 닫히지 않게 하려고).

In [ ]:
print("서버를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
client = MultiServerMCPClient({"search": WEB_SEARCH})
tools = await client.get_tools(server_name="search")
by_name = {tool.name: tool for tool in tools}   # 이름으로 꺼내 쓰려고 딕셔너리로

# 이 서버가 내주는 도구를 '이름(인자): 설명 첫 줄' 로 찍어 무엇을 할 수 있는지 확인한다.
print(f"도구 {len(tools)}개")
for tool in tools:
    print(f" - {tool.name}({', '.join(tool.args)}): {tool.description.strip().splitlines()[0][:60]}")

> 도구가 **둘**입니다. `search` 는 제목·요약·링크 **목록**만 주고, 본문은 주지 않습니다.
> 실제 내용이 필요하면 `fetch_content` 로 **한 번 더** 가져와야 합니다.
> 이 "목록 먼저, 본문은 필요할 때" 구조는 검색 서버 대부분이 같습니다. 한 번에 다 가져오면 느리고 비싸기 때문입니다.

---
## 2. 검색 도구를 직접 호출하기

`max_results` 를 작게 주는 데는 이유가 있습니다. 결과가 길수록 **나중에 에이전트가 읽을 토큰(=비용)** 도 커집니다.
사람이 보려고 검색하는 게 아니라 **모델에게 읽힐 자료**를 뽑는 것이므로, 필요한 만큼만 받습니다.

In [ ]:
found = await by_name["search"].ainvoke({"query": "MCP Model Context Protocol 개념", "max_results": 3})
print(block_text(found)[:800])

---
## 3. 본문 가져오기

검색 결과의 요약만으로는 근거가 얕습니다. `fetch_content` 에 URL 하나를 주면 그 페이지의 **본문 텍스트**를 돌려줍니다.

In [ ]:
body = block_text(await by_name["fetch_content"].ainvoke(
    {"url": "https://modelcontextprotocol.io/docs/getting-started/intro"}))

print("가져온 글자 수:", len(body))     # 본문이 통째로 오므로 길다. 이 길이가 곧 모델이 읽을 양이다.
print(body[:500])

### 🖐️ 함께 따라하기: 다른 주제를 검색하고 본문까지 이어 읽기

데모는 MCP 를 검색했습니다. 이번엔 **다른 주제**로 같은 두 도구를 이어 붙여 봅니다.

1. `search` 도구로 **`"LangChain agent middleware"`** 를 `max_results=2` 로 검색해 결과를 출력하세요.
2. 그 출력에서 URL 하나를 골라 `fetch_content` 로 본문을 가져오세요.
3. 본문의 **글자 수**와 **앞 300자**를 출력하세요.

**확인 기준**: 검색 결과에 제목과 링크가 나오고, 가져온 본문의 글자 수가 0 보다 큽니다.
URL 을 손으로 지어내지 말고 **1번 출력에 실제로 있던 주소**를 쓰세요. 이것이 다음 절에서 에이전트에게 요구할 태도와 같습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) search 도구로 "LangChain agent middleware" 를 max_results=2 로 검색해 출력한다
# 2) 그 결과에 나온 URL 하나를 골라 fetch_content 로 본문을 가져온다
# 3) 본문의 글자 수와 앞 300자를 출력한다

---
## 4. 검색 도구를 에이전트에 붙이기

여기서부터는 **우리가 도구를 부르지 않습니다**. 무엇을 부를지 모델이 정합니다.
시스템 프롬프트에서 두 가지를 못 박습니다.

1. **언제 도구를 쓸지**: 모델은 아는 척 답하려는 경향이 있어, 도구를 쥐여 줘도 안 쓰는 일이 있습니다.
2. **근거를 어떻게 남길지**: 문장마다 출처 URL 을 적게 해야 읽는 사람이 사실을 되짚을 수 있고,
   검색 결과에 없는 내용을 지어냈는지도 드러납니다.

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model,
    tools,   # MCP 도구를 그대로 넘긴다. 우리가 만든 도구와 같은 자리다
    system_prompt=(
        "너는 조사 담당이다. 사실 확인이 필요하면 반드시 search 도구로 검색하고, "
        "요약만으로 부족하면 fetch_content 로 본문까지 읽어 확인한 뒤 답한다. "
        "답의 문장마다 근거가 된 페이지의 URL 을 괄호로 붙이고, 맨 끝에 '참고한 주소' 목록을 "
        "'- 제목: URL' 형태로 정리한다. 검색 결과에 없는 내용은 쓰지 않고, "
        "확인하지 못한 부분은 확인하지 못했다고 밝힌다."
    ),
)
print("에이전트 준비 완료")

도구 하나라도 MCP 면 **에이전트도 `ainvoke`** 로 불러야 합니다.
질문 문자열만 넘기면 LangChain 이 사람 메시지로 바꿔 줍니다.

In [ ]:
question = "Model Context Protocol 은 무엇이고 어디에 쓰나요? 웹에서 찾아 3문장으로 정리해 주세요."
print("질문:", question, "\n")

# 메시지 기록을 함께 찍는다. 모델이 검색을 실제로 했는지는 이 기록으로만 확인할 수 있다.
result = await agent.ainvoke({"messages": question})
print_trajectory(result)

### 🖐️ 함께 따라하기: 도구를 쥐여 주지 않으면 어떻게 답하나

앞에서 만든 에이전트는 검색을 했습니다. 정말 도구 때문인지 확인하려면 **도구 없는 에이전트**와 비교해야 합니다.

1. `create_agent(model, [], system_prompt="너는 조사 담당이다. 근거 URL 을 붙여 답하라.")` 로 **도구가 빈** 에이전트를 만드세요.
2. 위와 **같은 질문**을 던져 답을 받으세요.
3. `print_trajectory()` 로 기록을 찍고, **도구 호출 줄이 있는지** 확인하세요.

**확인 기준**: 도구 호출 줄이 **하나도 없고**, 그런데도 답은 그럴듯하게 나옵니다.
URL 이 붙어 있다면 그것은 **모델이 기억에서 지어낸 주소**일 수 있습니다. 열어 보면 없는 페이지일 때도 있습니다.
이 차이가 "도구를 붙인다"는 말의 실제 의미입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 도구를 빈 리스트로 넘긴 에이전트를 만든다
# 2) 위와 같은 질문을 던진다
# 3) print_trajectory 로 기록을 찍어 도구 호출이 있는지 확인한다

---
## 이번 실습 정리

| 배운 것 | 요점 |
|---|---|
| 실행기 선택 | Node 패키지는 `npx`, 파이썬 패키지는 `uvx`. 인자 규칙은 **서버가 정한다** |
| 검색 도구 구조 | `search`(목록) → `fetch_content`(본문). 필요할 때만 본문을 받는다 |
| 비용 감각 | `max_results` 는 곧 모델이 읽을 토큰 양이다 |
| 에이전트에 붙이기 | MCP 도구를 `create_agent` 에 그대로 넘긴다. 도구가 MCP 면 에이전트도 `ainvoke` |
| 근거 남기기 | 시스템 프롬프트로 "검색해라 + URL 을 붙여라 + 모르면 모른다고 해라" 를 못 박는다 |

다음 실습: `03_브라우저조작_Playwright.ipynb` 에서 **상태를 가진 서버**(열어 둔 페이지가 남는 서버)를 다룹니다.